# LightGBM + XGBoost 앙상블 모델링 v1

## 구성
| 단계 | 내용 |
|------|------|
| 1 | 데이터 로드 & v2 피처 엔지니어링 |
| 2 | LightGBM 학습 (품목별) |
| 3 | XGBoost 학습 (품목별) |
| 4 | 앙상블 Validation MAE 비교 |
| 5 | Test 예측 (자기회귀 3단계) & Submission 저장 |

**평가 지표**: MAE  
**검증 방식**: 품목별 마지막 3순 시계열 holdout  
**앙상블**: LightGBM 50% + XGBoost 50% (단순 평균)

In [12]:
pip install optuna

In [13]:
import numpy as np
import pandas as pd
import glob, os, warnings
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

# ── 설정 ─────────────────────────────────────────────────────
TRAIN_PATH = 'train_target_only_cleaned (1).csv'
TEST_DIR   = 'test'
SAVE_PATH  = 'ensemble_lgbm_xgb_submission.csv'

USE_LOG    = False   # True → log1p 변환 타깃
VALID_SIZE = 9       # 마지막 9순(3개월치)을 validation으로
N_TRIALS   = 30      # Optuna 탐색 횟수 (품목당)
W_TRIALS   = 20      # 앙상블 가중치 탐색 횟수

GROUP_COLS = ['품목명', '품종명', '거래단위', '등급']
CAT_COLS   = ['품종명', '거래단위', '등급']
TARGET_COL = '평균가격(원)'
NORMAL_COL = '평년 평균가격(원)'
TIME_COL   = '시점'
순_map      = {'상순': 0, '중순': 1, '하순': 2}

---
## 1. 데이터 로드 & v2 피처 엔지니어링

In [14]:
df = pd.read_csv(TRAIN_PATH)
print('원본 shape:', df.shape)

# ── 시간 파싱 ─────────────────────────────────────────────────
df['year']     = df[TIME_COL].str[:4].astype(int)
df['month']    = df[TIME_COL].str[4:6].astype(int)
df['순_str']   = df[TIME_COL].str[6:]
df['순_num']   = df['순_str'].map(순_map)
df['time_idx'] = df['year'] * 36 + (df['month'] - 1) * 3 + df['순_num']

df = df.sort_values(GROUP_COLS + ['time_idx']).reset_index(drop=True)
df['y'] = np.log1p(df[TARGET_COL]) if USE_LOG else df[TARGET_COL]

print(f'time_idx 범위: {df["time_idx"].min()} ~ {df["time_idx"].max()}')
print(f'연도: {sorted(df["year"].unique())}')
print(f'품목: {sorted(df["품목명"].unique())}')
df.head(3)

원본 shape: (1425, 7)
time_idx 범위: 72648 ~ 72791
연도: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
품목: ['감자', '건고추', '깐마늘(국산)', '대파', '무', '배', '배추', '사과', '상추', '양파']


,시점,품목명,품종명,거래단위,등급,평년 평균가격(원),평균가격(원),year,month,순_str,순_num,time_idx,y
0,201801상순,감자,감자 수미,20키로상자,상,24660.031746,44170.285714,2018,1,상순,0,72648,44170.285714
1,201801중순,감자,감자 수미,20키로상자,상,23299.444444,48283.777778,2018,1,중순,1,72649,48283.777778
2,201801하순,감자,감자 수미,20키로상자,상,25218.007407,50243.000000,2018,1,하순,2,72650,50243.000000


In [15]:
g = df.groupby(GROUP_COLS)['y']

# ── [기존] Lag 1~3 ────────────────────────────────────────────
for lag in [1, 2, 3]:
    df[f'lag{lag}'] = g.shift(lag)

df['diff1']  = df['lag1'] - df['lag2']
df['diff2']  = df['lag2'] - df['lag3']
df['ratio1'] = df['lag1'] / (df['lag2'] + 1e-6)
df['ratio2'] = df['lag2'] / (df['lag3'] + 1e-6)

df['rolling_mean_3'] = g.transform(lambda x: x.shift(1).rolling(3).mean())
df['sin_month']      = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month']      = np.cos(2 * np.pi * df['month'] / 12)
df['normal_ratio']   = (df[TARGET_COL] - df[NORMAL_COL]) / (df[NORMAL_COL] + 1e-6)

# ── [추가] 중기·전년 Lag ──────────────────────────────────────
for lag in [6, 12, 36]:
    df[f'lag{lag}'] = g.shift(lag)

df['yoy_change'] = (df['lag1'] - df['lag36']) / (df['lag36'] + 1e-6)

# ── [추가] 롤링·EWM ──────────────────────────────────────────
df['rolling_mean_6'] = g.transform(lambda x: x.shift(1).rolling(6).mean())
df['rolling_std_3']  = g.transform(lambda x: x.shift(1).rolling(3).std())
df['rolling_std_6']  = g.transform(lambda x: x.shift(1).rolling(6).std())
df['ewm_span6']      = g.transform(lambda x: x.shift(1).ewm(span=6, adjust=False).mean())

# ── [추가] 변동성·모멘텀 ─────────────────────────────────────
_rm6 = g.transform(lambda x: x.shift(1).rolling(6).mean())
_rs6 = g.transform(lambda x: x.shift(1).rolling(6).std())
df['rolling_cv6']    = _rs6 / (_rm6 + 1e-6)
df['price_momentum'] = df['lag1'] / (df['lag3'] + 1e-6) - 1

_item_mean = df.groupby('품목명')['y'].transform('mean')
_item_std  = df.groupby('품목명')['y'].transform('std')
df['price_zscore'] = (df['lag1'] - _item_mean) / (_item_std + 1e-6)

# ── [추가] 시간·이벤트 ───────────────────────────────────────
df['sun_sin']    = np.sin(2 * np.pi * df['순_num'] / 3)
df['sun_cos']    = np.cos(2 * np.pi * df['순_num'] / 3)
df['year_trend'] = df['year'] - 2018
df['is_2020']    = (df['year'] == 2020).astype(int)

# ── [추가] 평년 대비 확장 ────────────────────────────────────
g_nr = df.groupby(GROUP_COLS)['normal_ratio']
df['normal_ratio_lag1']  = g_nr.shift(1)
df['normal_ratio_roll3'] = g_nr.transform(lambda x: x.shift(1).rolling(3).mean())
df['is_above_normal']    = (df['normal_ratio'] > 0).astype(int)
df['log_normal_price']   = np.log1p(df[NORMAL_COL])

print('피처 엔지니어링 완료, shape:', df.shape)

피처 엔지니어링 완료, shape: (1425, 43)


In [16]:
# ── 카테고리 인코딩 (LightGBM용) ─────────────────────────────
for c in CAT_COLS:
    df[c] = df[c].astype('category')

# ── 품목별 통계 저장 (test price_zscore 계산용) ───────────────
item_stats = (
    df.groupby('품목명')['y']
    .agg(['mean', 'std'])
    .to_dict('index')
)

# ── 피처 목록 ────────────────────────────────────────────────
FEATURE_COLS = [
    # 카테고리
    '품종명', '거래단위', '등급',
    # [기존] lag
    'lag1', 'lag2', 'lag3',
    # [기존] 차분·비율
    'diff1', 'diff2', 'ratio1', 'ratio2',
    # [기존] 롤링
    'rolling_mean_3',
    # [기존] 시간
    'month', '순_num', 'sin_month', 'cos_month', 'time_idx',
    # [기존] 평년
    'normal_ratio', 'log_normal_price',
    # [추가] 중기·전년 lag
    'lag6', 'lag12', 'lag36', 'yoy_change',
    # [추가] 롤링·EWM
    'rolling_mean_6', 'rolling_std_3', 'rolling_std_6', 'ewm_span6',
    # [추가] 변동성·모멘텀
    'rolling_cv6', 'price_momentum', 'price_zscore',
    # [추가] 시간·이벤트
    'sun_sin', 'sun_cos', 'year_trend', 'is_2020',
    # [추가] 평년 대비 확장
    'normal_ratio_lag1', 'normal_ratio_roll3', 'is_above_normal',
]

df_model = df.dropna(subset=['lag1', 'lag2', 'lag3']).copy()
print(f'학습 데이터: {df_model.shape[0]}행 × {len(FEATURE_COLS)}개 피처')
print(f'피처 결측 현황:')
print(df_model[FEATURE_COLS].isna().sum()[df_model[FEATURE_COLS].isna().sum() > 0])

학습 데이터: 1392행 × 36개 피처
피처 결측 현황:
lag6               33
lag12              99
lag36             346
yoy_change        346
rolling_mean_6     33
rolling_std_6      33
rolling_cv6        33
dtype: int64


---
## 2. LightGBM 학습 (Optuna 하이퍼파라미터 튜닝)

품목(품목명)별로 개별 모델 학습.  
마지막 3순을 Validation holdout → Optuna로 MAE 최소화 파라미터 탐색.

In [17]:
def time_holdout_split(item_df, valid_size=VALID_SIZE):
    item_df = item_df.sort_values('time_idx').reset_index(drop=True)
    if len(item_df) <= valid_size:
        return None, None
    return item_df.iloc[:-valid_size].copy(), item_df.iloc[-valid_size:].copy()


def tune_lgb(item, n_trials=N_TRIALS):
    item_df = df_model[df_model['품목명'] == item].copy()
    tr, va  = time_holdout_split(item_df)
    if tr is None:
        return None, None

    X_tr, y_tr = tr[FEATURE_COLS], tr['y']
    X_va, y_va = va[FEATURE_COLS], va['y']

    def objective(trial):
        params = {
            'objective'        : 'regression',
            'metric'           : 'mae',
            'verbosity'        : -1,
            'n_estimators'     : trial.suggest_int('n_estimators', 300, 1500),
            'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves'       : trial.suggest_int('num_leaves', 16, 128),
            'max_depth'        : trial.suggest_int('max_depth', 3, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state'     : 42,
        }
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_va, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(-1)])
        return mean_absolute_error(y_va, model.predict(X_va))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    model = LGBMRegressor(**best, objective='regression', metric='mae',
                          verbosity=-1, random_state=42)
    model.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                          lgb.log_evaluation(-1)])
    mae = mean_absolute_error(y_va, model.predict(X_va))
    return model, mae


lgb_models, lgb_val_maes = {}, {}

for item in sorted(df_model['품목명'].unique()):
    model, mae = tune_lgb(item)
    if model is None:
        print(f'[LGB] {item} → 데이터 부족, skip')
        continue
    lgb_models[item]   = model
    lgb_val_maes[item] = mae
    print(f'[LGB] {item:15s} | val MAE: {mae:>10,.1f}')

print(f'\n[LGB] 전체 평균 MAE: {np.mean(list(lgb_val_maes.values())):,.1f}')

[LGB] 감자              | val MAE:      976.1
[LGB] 건고추             | val MAE:    5,195.5
[LGB] 깐마늘(국산)         | val MAE:    1,504.5
[LGB] 대파              | val MAE:       56.8
[LGB] 무               | val MAE:       51.5
[LGB] 배               | val MAE:      574.8
[LGB] 배추              | val MAE:      754.0
[LGB] 사과              | val MAE:      353.9
[LGB] 상추              | val MAE:      138.7
[LGB] 양파              | val MAE:       20.8

[LGB] 전체 평균 MAE: 962.6


---
## 3. XGBoost 학습 (Optuna 하이퍼파라미터 튜닝)

XGBoost는 category dtype 미지원 → LabelEncoder로 인코딩 후 Optuna 탐색.

In [18]:
# XGBoost용 Label Encoding
le_dict = {}
df_xgb  = df_model.copy()

for c in CAT_COLS:
    le = LabelEncoder()
    df_xgb[c] = le.fit_transform(df_xgb[c].astype(str))
    le_dict[c] = le


def tune_xgb(item, n_trials=N_TRIALS):
    item_df = df_xgb[df_xgb['품목명'] == item].copy()
    tr, va  = time_holdout_split(item_df)
    if tr is None:
        return None, None

    X_tr, y_tr = tr[FEATURE_COLS], tr['y']
    X_va, y_va = va[FEATURE_COLS], va['y']

    def objective(trial):
        params = {
            'objective'       : 'reg:absoluteerror',
            'tree_method'     : 'hist',
            'verbosity'       : 0,
            'random_state'    : 42,
            'n_estimators'    : trial.suggest_int('n_estimators', 300, 1500),
            'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth'       : trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
            'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }
        model = XGBRegressor(**params, early_stopping_rounds=50)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return mean_absolute_error(y_va, model.predict(X_va))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    model = XGBRegressor(**best, objective='reg:absoluteerror', tree_method='hist',
                         verbosity=0, random_state=42, early_stopping_rounds=50)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    mae = mean_absolute_error(y_va, model.predict(X_va))
    return model, mae


xgb_models, xgb_val_maes = {}, {}

for item in sorted(df_xgb['품목명'].unique()):
    model, mae = tune_xgb(item)
    if model is None:
        print(f'[XGB] {item} → 데이터 부족, skip')
        continue
    xgb_models[item]   = model
    xgb_val_maes[item] = mae
    print(f'[XGB] {item:15s} | val MAE: {mae:>10,.1f}')

print(f'\n[XGB] 전체 평균 MAE: {np.mean(list(xgb_val_maes.values())):,.1f}')

[XGB] 감자              | val MAE:    1,189.4
[XGB] 건고추             | val MAE:    3,999.9
[XGB] 깐마늘(국산)         | val MAE:    1,793.1
[XGB] 대파              | val MAE:       48.2
[XGB] 무               | val MAE:      117.4
[XGB] 배               | val MAE:      626.3
[XGB] 배추              | val MAE:      763.7
[XGB] 사과              | val MAE:      373.0
[XGB] 상추              | val MAE:      151.5
[XGB] 양파              | val MAE:       13.1

[XGB] 전체 평균 MAE: 907.6


In [18]:
# XGBoost용 Label Encoding
le_dict = {}
df_xgb  = df_model.copy()

for c in CAT_COLS:
    le = LabelEncoder()
    df_xgb[c] = le.fit_transform(df_xgb[c].astype(str))
    le_dict[c] = le


def tune_xgb(item, n_trials=N_TRIALS):
    item_df = df_xgb[df_xgb['품목명'] == item].copy()
    tr, va  = time_holdout_split(item_df)
    if tr is None:
        return None, None

    X_tr, y_tr = tr[FEATURE_COLS], tr['y']
    X_va, y_va = va[FEATURE_COLS], va['y']

    def objective(trial):
        params = {
            'objective'       : 'reg:absoluteerror',
            'tree_method'     : 'hist',
            'verbosity'       : 0,
            'random_state'    : 42,
            'n_estimators'    : trial.suggest_int('n_estimators', 300, 1500),
            'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth'       : trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
            'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }
        model = XGBRegressor(**params, early_stopping_rounds=50)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return mean_absolute_error(y_va, model.predict(X_va))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    model = XGBRegressor(**best, objective='reg:absoluteerror', tree_method='hist',
                         verbosity=0, random_state=42, early_stopping_rounds=50)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    mae = mean_absolute_error(y_va, model.predict(X_va))
    return model, mae


xgb_models, xgb_val_maes = {}, {}

for item in sorted(df_xgb['품목명'].unique()):
    model, mae = tune_xgb(item)
    if model is None:
        print(f'[XGB] {item} → 데이터 부족, skip')
        continue
    xgb_models[item]   = model
    xgb_val_maes[item] = mae
    print(f'[XGB] {item:15s} | val MAE: {mae:>10,.1f}')

print(f'\n[XGB] 전체 평균 MAE: {np.mean(list(xgb_val_maes.values())):,.1f}')

[XGB] 감자              | val MAE:    1,189.4
[XGB] 건고추             | val MAE:    3,999.9
[XGB] 깐마늘(국산)         | val MAE:    1,793.1
[XGB] 대파              | val MAE:       48.2
[XGB] 무               | val MAE:      117.4
[XGB] 배               | val MAE:      626.3
[XGB] 배추              | val MAE:      763.7
[XGB] 사과              | val MAE:      373.0
[XGB] 상추              | val MAE:      151.5
[XGB] 양파              | val MAE:       13.1

[XGB] 전체 평균 MAE: 907.6


---
## 5. Test 예측 (자기회귀 3단계 앙상블)

각 테스트 파일(TEST_00 ~ TEST_24)에 대해:
1. T-8순 ~ T+0순 이력 + 학습 데이터 이력 결합
2. T+1, T+2, T+3 순서로 자기회귀 예측
3. LightGBM과 XGBoost 예측 가중 평균

In [20]:
# ── 학습 데이터 원본 이력 보존 (combined history 구성용) ───────
df_hist = df.copy()

def parse_test_rel(시점_str):
    """T-8순 → -8, T+0순 → 0"""
    s = str(시점_str).strip()
    if 'T+' in s:
        return int(s.replace('T+', '').replace('순', ''))
    elif 'T-' in s:
        return -int(s.replace('T-', '').replace('순', ''))
    return 0

def advance_time(year, month, sun):
    """현재 시점에서 1순 전진"""
    sun += 1
    if sun == 3:
        sun, month = 0, month + 1
    if month == 13:
        month, year = 1, year + 1
    return year, month, sun

def compute_ewm(history, span=6):
    """지수가중이동평균 (adjust=False, 전체 이력 사용)"""
    if not history:
        return np.nan
    alpha = 2 / (span + 1)
    ewm = history[0]
    for v in history[1:]:
        ewm = alpha * v + (1 - alpha) * ewm
    return ewm

def build_feature_row(history, normal_price, year, month, sun_num,
                      품종명, 거래단위, 등급, item, for_xgb=False):
    """단일 예측 시점의 피처 행 생성"""
    n = len(history)
    lag = lambda k: history[-k] if n >= k else np.nan

    lag1, lag2, lag3   = lag(1), lag(2), lag(3)
    lag6, lag12, lag36 = lag(6), lag(12), lag(36)

    def safe_mean(lst): return float(np.nanmean(lst)) if lst else np.nan
    def safe_std(lst):  return float(np.nanstd(lst))  if len(lst) > 1 else np.nan

    r3, r6 = history[-3:] if n >= 3 else history, history[-6:] if n >= 6 else history

    diff1    = (lag1 - lag2) if not (np.isnan(lag1) or np.isnan(lag2)) else np.nan
    diff2    = (lag2 - lag3) if not (np.isnan(lag2) or np.isnan(lag3)) else np.nan
    ratio1   = lag1 / (lag2 + 1e-6)
    ratio2   = lag2 / (lag3 + 1e-6)
    rm3, rm6 = safe_mean(r3), safe_mean(r6)
    rs3, rs6 = safe_std(r3),  safe_std(r6)
    cv6      = rs6 / (rm6 + 1e-6)
    ewm6     = compute_ewm(history)
    momentum = lag1 / (lag3 + 1e-6) - 1
    yoy      = (lag1 - lag36) / (lag36 + 1e-6) if not np.isnan(lag36) else np.nan

    i_s    = item_stats.get(item, {'mean': 0, 'std': 1})
    zscore = (lag1 - i_s['mean']) / (i_s['std'] + 1e-6)

    sin_m  = np.sin(2 * np.pi * month   / 12)
    cos_m  = np.cos(2 * np.pi * month   / 12)
    sin_s  = np.sin(2 * np.pi * sun_num / 3)
    cos_s  = np.cos(2 * np.pi * sun_num / 3)

    # normal_ratio: 실제가격 기준 (USE_LOG=False 기준)
    lag1_raw = float(np.expm1(lag1)) if USE_LOG else lag1
    lag2_raw = float(np.expm1(lag2)) if USE_LOG else lag2
    nr       = (lag1_raw - normal_price) / (normal_price + 1e-6)
    nr_lag1  = (lag2_raw - normal_price) / (normal_price + 1e-6) if not np.isnan(lag2) else np.nan
    nr_r3    = [(float(np.expm1(h)) if USE_LOG else h - normal_price) / (normal_price + 1e-6) for h in r3]
    nr_roll3 = safe_mean(nr_r3)

    row = {
        '품종명': 품종명, '거래단위': 거래단위, '등급': 등급,
        'lag1': lag1, 'lag2': lag2, 'lag3': lag3,
        'diff1': diff1, 'diff2': diff2,
        'ratio1': ratio1, 'ratio2': ratio2,
        'rolling_mean_3': rm3,
        'month': month, '순_num': sun_num,
        'sin_month': sin_m, 'cos_month': cos_m,
        'time_idx': year * 36 + (month - 1) * 3 + sun_num,
        'normal_ratio': nr, 'log_normal_price': np.log1p(normal_price),
        'lag6': lag6, 'lag12': lag12, 'lag36': lag36, 'yoy_change': yoy,
        'rolling_mean_6': rm6, 'rolling_std_3': rs3, 'rolling_std_6': rs6,
        'ewm_span6': ewm6, 'rolling_cv6': cv6,
        'price_momentum': momentum, 'price_zscore': zscore,
        'sun_sin': sin_s, 'sun_cos': cos_s,
        'year_trend': year - 2018, 'is_2020': int(year == 2020),
        'normal_ratio_lag1': nr_lag1,
        'normal_ratio_roll3': nr_roll3,
        'is_above_normal': int(nr > 0),
    }

    X = pd.DataFrame([row])
    if for_xgb:
        for c in CAT_COLS:
            val = str(X[c].iloc[0])
            X[c] = le_dict[c].transform([val])[0] if val in le_dict[c].classes_ else -1
    else:
        for c in CAT_COLS:
            X[c] = X[c].astype('category')

    return X[FEATURE_COLS]

In [21]:
def predict_test_file(test_path):
    test    = pd.read_csv(test_path)
    test_id = os.path.splitext(os.path.basename(test_path))[0]
    test['rel_idx'] = test[TIME_COL].apply(parse_test_rel)
    test = test.sort_values('rel_idx')

    pred_by_step = {1: {}, 2: {}, 3: {}}

    for keys, grp in test.groupby(GROUP_COLS):
        품목명_k, 품종명_k, 거래단위_k, 등급_k = keys
        grp = grp.sort_values('rel_idx').reset_index(drop=True)

        normal_price = float(grp[NORMAL_COL].iloc[-1])
        test_hist_y  = grp[TARGET_COL].tolist()

        # 학습 데이터 해당 그룹 이력
        mask = (
            (df_hist['품목명']   == 품목명_k) &
            (df_hist['품종명']   == 품종명_k) &
            (df_hist['거래단위'] == 거래단위_k) &
            (df_hist['등급']     == 등급_k)
        )
        train_grp = df_hist[mask].sort_values('time_idx')
        if len(train_grp) == 0:
            continue

        # 캘린더 기준: 학습 마지막 시점 = T+0
        last   = train_grp.iloc[-1]
        base_y = int(last['year'])
        base_m = int(last['month'])
        base_s = int(last['순_num'])

        # combined history: 학습 y 뒤에 테스트 이력 append
        train_y   = train_grp['y'].tolist()
        test_y    = [np.log1p(v) if USE_LOG else v for v in test_hist_y]
        combined  = train_y + test_y

        # T+1부터 예측 (학습 마지막 기준 1순 앞)
        cy, cm, cs = advance_time(base_y, base_m, base_s)

        # 품목별 최적 가중치 사용
        w_lgb = best_weights.get(품목명_k, 0.5)
        w_xgb = 1.0 - w_lgb

        current = combined.copy()
        for step in range(1, 4):
            X_lgb = build_feature_row(
                current, normal_price, cy, cm, cs,
                품종명_k, 거래단위_k, 등급_k, 품목명_k, for_xgb=False
            )
            X_xgb = build_feature_row(
                current, normal_price, cy, cm, cs,
                품종명_k, 거래단위_k, 등급_k, 품목명_k, for_xgb=True
            )

            p_lgb = float(max(0.0, lgb_models[품목명_k].predict(X_lgb)[0])) if 품목명_k in lgb_models else 0.0
            p_xgb = float(max(0.0, xgb_models[품목명_k].predict(X_xgb)[0])) if 품목명_k in xgb_models else 0.0
            p_ens = w_lgb * p_lgb + w_xgb * p_xgb
            p_raw = float(np.expm1(p_ens) if USE_LOG else p_ens)

            if 품목명_k not in pred_by_step[step]:
                pred_by_step[step][품목명_k] = []
            pred_by_step[step][품목명_k].append(p_raw)

            current.append(p_ens)
            cy, cm, cs = advance_time(cy, cm, cs)

    # 결과 DataFrame: 3행 (step 1/2/3) × 품목 열
    all_items = sorted(test['품목명'].unique())
    rows = []
    for step in [1, 2, 3]:
        row = {'시점': f'{test_id}+{step}순'}
        for item in all_items:
            vals = pred_by_step[step].get(item, [0.0])
            row[item] = float(np.mean(vals)) if vals else 0.0
        rows.append(row)

    return pd.DataFrame(rows)

---
## 6. Submission 생성

In [22]:
sample_sub = pd.read_csv('sample_submission.csv')
test_files = sorted(glob.glob(os.path.join(TEST_DIR, 'TEST_*.csv')))
print(f'Test 파일 수: {len(test_files)}개\n')

pred_dfs = []
for tf in test_files:
    df_pred = predict_test_file(tf)
    pred_dfs.append(df_pred)
    print(f'완료: {os.path.basename(tf)}')

submission = pd.concat(pred_dfs, ignore_index=True)

# sample_submission 기준으로 정렬 & 컬럼 맞추기
submission = sample_sub[['시점']].merge(submission, on='시점', how='left')
for col in submission.columns[1:]:
    submission[col] = submission[col].fillna(0).clip(lower=0)

# 누락 컬럼 보정
for col in sample_sub.columns:
    if col not in submission.columns:
        submission[col] = 0
submission = submission[sample_sub.columns]

submission.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print(f'\n저장 완료: {SAVE_PATH}  ({submission.shape[0]}행 × {submission.shape[1]}열)')
submission.head(10)

Test 파일 수: 25개

완료: TEST_00.csv
완료: TEST_01.csv
완료: TEST_02.csv
완료: TEST_03.csv
완료: TEST_04.csv
완료: TEST_05.csv
완료: TEST_06.csv
완료: TEST_07.csv
완료: TEST_08.csv
완료: TEST_09.csv
완료: TEST_10.csv
완료: TEST_11.csv
완료: TEST_12.csv
완료: TEST_13.csv
완료: TEST_14.csv
완료: TEST_15.csv
완료: TEST_16.csv
완료: TEST_17.csv
완료: TEST_18.csv
완료: TEST_19.csv
완료: TEST_20.csv
완료: TEST_21.csv
완료: TEST_22.csv
완료: TEST_23.csv
완료: TEST_24.csv

저장 완료: ensemble_lgbm_xgb_submission.csv  (75행 × 11열)


,시점,감자,건고추,깐마늘(국산),대파,무,배추,사과,상추,양파,배
0,TEST_00+1순,42559.628873,613204.349432,155109.971334,1613.830862,22261.301996,13534.791890,25346.079078,1256.537882,1489.579647,30314.680464
1,TEST_00+2순,44567.091151,599325.170971,154363.839045,1462.197637,22254.697701,12720.886205,25105.833090,1357.867382,1470.591372,30185.361386
2,TEST_00+3순,44670.988878,598201.379911,154366.785105,1308.772489,22165.978408,13138.761093,25214.533488,1449.245875,1463.287812,30082.058611
3,TEST_01+1순,44369.869501,609482.286375,155441.075437,1795.715439,13609.943230,5404.619936,25116.033430,1004.818209,1450.167630,28891.549460
4,TEST_01+2순,46344.462770,599325.170971,154366.151243,1834.808626,12828.574951,4896.290827,25183.498386,1034.397556,1450.192703,29158.063647
5,TEST_01+3순,51692.877934,598816.853368,154365.798969,1937.548585,12577.718748,4548.755930,25336.202949,1079.654989,1454.887403,29176.448886
6,TEST_02+1순,50324.810626,558871.109511,155800.135131,1281.535648,9961.597229,10550.139700,25694.303170,1253.236550,570.151206,36073.043564
7,TEST_02+2순,43979.995590,560281.733963,154367.104783,1352.962710,10255.464427,10402.167242,25684.788189,1306.891059,588.839952,34845.037735
8,TEST_02+3순,37497.393214,561476.528973,154366.728913,1480.435005,10706.540825,10540.827248,25684.788189,1249.984824,601.501466,34276.643411
9,TEST_03+1순,50634.345335,558871.109511,155793.374647,1608.840226,13751.405750,8034.854569,25694.303170,901.576458,806.910700,35276.532997
